# Import

In [ ]:
import os
import psycopg
import pandas as pd
from sqlalchemy import create_engine, text

# ETL

In [ ]:
!psql $DATABASE_URL -f ../sql/ddl.sql
!psql $DATABASE_URL -f ../sql/load.sql
!psql $DATABASE_URL -f ../sql/transform.sql

psql:../sql/ddl.sql:11: NOTICE:  relation "train" already exists, skipping
CREATE TABLE
psql:../sql/ddl.sql:21: NOTICE:  relation "client" already exists, skipping
CREATE TABLE
psql:../sql/ddl.sql:29: NOTICE:  relation "gas_prices" already exists, skipping
CREATE TABLE
psql:../sql/ddl.sql:36: NOTICE:  relation "electricity_prices" already exists, skipping
CREATE TABLE
TRUNCATE TABLE
TRUNCATE TABLE
TRUNCATE TABLE
TRUNCATE TABLE
COPY 2018352
COPY 41919
COPY 637
COPY 15286
DROP TABLE
SELECT 2018352


# Queries

In [ ]:
db_url = os.environ["DATABASE_URL"]
engine = create_engine(
    db_url.replace("postgresql://", "postgresql+psycopg://")
)

In [ ]:
pd.read_sql(
    """
SELECT *
FROM merged
ORDER BY
    datetime,
    county,
    product_type,
    is_business,
    is_consumption,
    target 
LIMIT 5;
""",
    engine,
)

,datetime,county,product_type,is_business,is_consumption,target,data_block_id,prediction_unit_id,eic_count,installed_capacity,lowest_price_per_mwh,highest_price_per_mwh,euros_per_mwh,row_id
0,2021-09-01,0,0,True,False,0.000,0,3,None,None,None,None,None,6
1,2021-09-01,0,0,True,True,59.000,0,3,None,None,None,None,None,7
2,2021-09-01,0,1,False,False,0.713,0,0,None,None,None,None,None,0
3,2021-09-01,0,1,False,True,96.590,0,0,None,None,None,None,None,1
4,2021-09-01,0,1,True,False,0.000,0,4,None,None,None,None,None,8


In [ ]:
# Checking nulls
pd.read_sql(
    """
SELECT
    COUNT(*) - COUNT(target) AS target_nulls,
    COUNT(*) FILTER (WHERE datetime IS NULL) AS datetime_nulls,
    COUNT(*) FILTER (WHERE eic_count IS NULL) AS eic_nulls,
    COUNT(*) FILTER (WHERE installed_capacity IS NULL) AS installed_capacity_nulls
FROM merged;
""",
    engine,
)

,target_nulls,datetime_nulls,eic_nulls,installed_capacity_nulls
0,528,0,8640,8640


- `target_nulls` - it's missing values that appear because of the shift from/to daylight saving time.
- `eic_nulls`, `installed_capacity_nulls` - missing values for the first days of the dataset, where client data is not yet available.

In [ ]:
# Checking completeness
pd.read_sql(
    """
SELECT
    COUNT(*) AS number_of_timeseries,
    MIN(number_of_entries) AS min_entries,
    AVG(number_of_entries) AS avg_entries,
    PERCENTILE_DISC(0.5) WITHIN GROUP (ORDER BY number_of_entries) AS median_entries,
    MAX(number_of_entries) AS max_entries
FROM (
    SELECT
        county,
        product_type,
        is_business,
        is_consumption,
        COUNT(target) AS number_of_entries
    FROM merged
    GROUP BY county, product_type, is_business, is_consumption
) AS timeseries_completeness;
""",
    engine,
)

,number_of_timeseries,min_entries,avg_entries,median_entries,max_entries
0,138,1656,14621.913043,15308,15308


There are 138 unique time series, most of them are complete (median = max = 15308). A few time series have significantly fewer entries (min = 1656), as explored in in "`1. Exploratory Data Analysis.ipynb`".

In [ ]:
category_list = ["is_business", "product_type", "county"]

for category in category_list:
    display(
        pd.read_sql(
            f"""
SELECT
    {category},
    is_consumption,
    AVG(target) AS hourly_avg_target,
    SUM(target) AS total_target
FROM merged
GROUP by is_consumption, {category} 
ORDER BY is_consumption, hourly_avg_target DESC 
""",
            engine,
        )
    )

,is_business,is_consumption,hourly_avg_target,total_target
0,False,False,98.959477,4.624386e+07
1,True,False,80.412651,4.355238e+07
2,True,True,744.642044,4.033063e+08
3,False,True,131.623055,6.150759e+07


,product_type,is_consumption,hourly_avg_target,total_target
0,3,False,152.741970,7.014522e+07
1,1,False,41.115457,1.606438e+07
2,0,False,37.289363,3.178918e+06
3,2,False,5.531463,4.077131e+05
4,3,True,809.279374,3.716535e+08
5,0,True,438.755381,3.740390e+07
6,1,True,136.351775,5.327455e+07
7,2,True,33.673468,2.482004e+06


,county,is_consumption,hourly_avg_target,total_target
0,0,False,256.191883,2.726804e+07
1,11,False,128.201636,1.268863e+07
2,14,False,99.474381,6.255745e+06
3,7,False,90.077297,7.793578e+06
4,10,False,82.642888,5.562032e+06
5,5,False,82.047097,6.218842e+06
6,3,False,61.193426,3.746996e+06
7,13,False,57.935266,3.505779e+06
8,9,False,56.699870,3.471846e+06
9,8,False,55.553943,2.551259e+06


- Business segment is consuming far more than non-business (745 vs 132 avg), which is explained by industrial facilities. For production the difference is much smaller: non-business prosumers produce at a comparable level (99 vs 80 avg).
- Product type 3 (spot contract) dominates in both categories (809 and 153 avg for consumption and production). Product type 2 (general service) has the smallest volumes of both production (6 avg) and consumption (34 avg).
- County 0 (Harjumaa) differs significantly from second place (Tartumaa) for both consumption (1626 vs 915) and production (256 vs 128 avg). County 12 (unknown, possibly intercounty or aggregated data) has the least avg production (6.5) but third place for consumption (547 avg). The least avg consumption is in county 1 (Hiiumaa, the smallest county in Estonia) - 48.

In [ ]:
# Top-5 aggregated consumption values for the last train month by county
# and is_business
pd.read_sql(
    """
SELECT
    county
    , is_business
    , SUM(target) AS consumption
FROM merged
WHERE is_consumption
    AND datetime >= '2023-03-01'
    AND datetime <'2023-04-01'
GROUP BY county, is_business
ORDER BY consumption DESC;
""",
    engine,
)

In [ ]:
# Consumption moving averages for business segment in the most consuming
# county
pd.read_sql(
    """
WITH daily_cons AS (
    SELECT DATE(datetime) AS date, SUM(TARGET) AS consumption
    FROM merged
    WHERE is_consumption
        AND county = 0
        AND is_business
    GROUP BY DATE(datetime)
),

with_ma AS (
    SELECT
        date,
        consumption,
        AVG(consumption) OVER (
            ORDER BY date
            ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
        ) AS consumption_ma_3d,
        AVG(consumption) OVER (
            ORDER BY date
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS consumption_ma_7d
    FROM daily_cons
)

SELECT date, consumption, consumption_ma_3d, consumption_ma_7d
FROM with_ma
WHERE date >= '2023-03-01' AND date < '2023-04-01';
""",
    engine,
)